# Data Preprocessing
Load the raw data, merge and clean it, impute genuine data gaps, and engineer features for both an XGBoost dataset and a logistic regression dataset with t+1 and t+3 prediction targets.

## Summary

**Data prep**
- Merged monthly `credit_applications` + `customers` data per client
- Detected genuine data gaps (missing months *after* a client's first appearance) vs. normal onboarding gaps
- Imputed genuine gaps using **only prior history** (medians, forward-filled CRG) — no future data leakage

**Engineered features** *(all computed using only past/current data per client)*
- `months_history_available` — client tenure
- `{col}_zscore` — deviation from client's historical average (extreme values capped, not dropped)
- `{col}_delta` — month-over-month % change
- `nr_credit_applications_past_sum` — cumulative prior applications
- `months_since_last_credit_application` — recency of last application

**Handling missing/invalid values**
- Zeros replaced with a small constant before ratio calculations, to avoid divide-by-zero
- Extreme/undefined values (`inf`) capped at a fixed bound instead of discarded
- Remaining `NaN`s filled with `0` — safe because tenure feature already flags "insufficient history"

**Model-ready outputs**
- `xgboost_data` — categorical features kept raw (trees handle natively)
- `log_reg_data` — categoricals one-hot encoded, fully numeric, no `NaN`/`inf`
- Each split into **t+1** and **t+3** prediction targets → **4 final datasets**

## Imports and display settings

In [45]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', lambda value: f'{value:,.2f}')
sns.set_theme(style='whitegrid', palette='deep')

## Load raw data
Load `credit_applications` and `customers`, and convert the `yearmonth` column to a monthly `Period`.

In [46]:
# Load the raw datasets, skipping the first column (index column from CSV export).
credit_applications = pd.read_csv('./data/credit_applications.csv', index_col=0)
customers = pd.read_csv('./data/customers.csv', index_col=0)

# Convert YYYYMM to a proper month-period value (year-month only, no day).
for dataframe in (credit_applications, customers):
    dataframe["yearmonth"] = pd.to_datetime(dataframe["yearmonth"].astype(str), format="%Y%m").dt.to_period("M")


display(credit_applications.head())
display(customers.head())

,client_nr,yearmonth,credit_application,nr_credit_applications
1,1,2014-01,0,0
2,1,2014-02,0,0
3,1,2014-03,0,0
4,1,2014-04,0,0
5,1,2014-05,0,0


,client_nr,yearmonth,total_nr_trx,nr_debit_trx,volume_debit_trx,nr_credit_trx,volume_credit_trx,min_balance,max_balance,CRG
1,1,2014-01,97,50,6527929,47,7454863,-7914288,25110651,1.00
2,1,2014-02,88,59,3475918,29,1895848,-8448513,25036651,1.00
3,1,2014-03,96,62,31316405,34,20083583,-10347650,18020151,1.00
4,1,2014-04,83,53,18669967,30,1091295,-15385039,13318200,1.00
5,1,2014-05,94,54,2893905,40,2034075,-15682170,2350000,1.00


## Merge datasets
Inner-join `credit_applications` and `customers` on `client_nr` and `yearmonth`.

In [47]:
# Merge credit_applications and customers on client_nr and yearmonth.
merged_data = credit_applications.merge(
    customers,
    on=['client_nr', 'yearmonth'],
    how='inner',
    validate='one_to_one',
    suffixes=('_application', '_customer')
)


display(merged_data.head())

,client_nr,yearmonth,credit_application,nr_credit_applications,total_nr_trx,nr_debit_trx,volume_debit_trx,nr_credit_trx,volume_credit_trx,min_balance,max_balance,CRG
0,1,2014-01,0,0,97,50,6527929,47,7454863,-7914288,25110651,1.00
1,1,2014-02,0,0,88,59,3475918,29,1895848,-8448513,25036651,1.00
2,1,2014-03,0,0,96,62,31316405,34,20083583,-10347650,18020151,1.00
3,1,2014-04,0,0,83,53,18669967,30,1091295,-15385039,13318200,1.00
4,1,2014-05,0,0,94,54,2893905,40,2034075,-15682170,2350000,1.00


## Sort chronologically
Sort by `client_nr` and `yearmonth` so later feature engineering steps can rely on temporal order.

In [48]:
# Sort by client_nr and yearmonth to ensure proper temporal ordering.
merged_data = merged_data.sort_values(['client_nr', 'yearmonth']).reset_index(drop=True)

## Detect genuine data gaps
For each client, distinguish onboarding gaps (missing months before the client's first appearance) from genuine data gaps (missing months after the client has already started appearing). Only genuine gaps will be imputed.

In [49]:
# Identify genuine missing months per client (gaps after they've started appearing)
# Distinguish from onboarding periods (missing months before first appearance)

# Extract unique client-month combinations and sort
client_months = merged_data[["client_nr", "yearmonth"]].drop_duplicates().copy()
client_months = client_months.sort_values(["client_nr", "yearmonth"]).reset_index(drop=True)

# For each client, determine if missing months occur only before first observation (onboarding)
# or if there are gaps after the client has already started producing records (true gap)
missing_pattern = []
for client_id, group in client_months.groupby("client_nr"):
    months = sorted(group["yearmonth"].unique())
    
    # Single-month or no-gap clients are not considered to have true gaps
    if len(months) <= 1:
        missing_pattern.append({
            "client_nr": client_id,
            "first_observed_month": months[0] if months else None,
            "has_true_gap": False,
            "reason": "single-month client",
            "missing_months_after_first_observation": [],
        })
        continue
    
    # Create a complete month range from first to last observed
    month_range = pd.period_range(start=months[0], end=months[-1], freq="M")
    observed_set = set(months)
    missing_all = sorted(set(month_range) - observed_set)
    
    # Classify missing months: before vs after first observation
    missing_before_first = [m for m in missing_all if m < months[0]]
    missing_after_first = [m for m in missing_all if m > months[0]]
    
    # True gap = has missing months after the client has already appeared
    has_true_gap = bool(missing_after_first)
    
    missing_pattern.append({
        "client_nr": client_id,
        "first_observed_month": months[0],
        "has_true_gap": has_true_gap,
        "reason": "onboarding gap" if not has_true_gap else "true data gap",
        "missing_months_after_first_observation": missing_after_first,
    })

# Convert to DataFrame for analysis
missing_pattern_df = pd.DataFrame(missing_pattern)

# Extract genuine gaps into a dictionary for later imputation
genuine_gaps_per_client = {
    row['client_nr']: row['missing_months_after_first_observation']
    for _, row in missing_pattern_df.iterrows()
    if row['has_true_gap']
}

# Summary statistics
true_gap_clients = missing_pattern_df[missing_pattern_df["has_true_gap"]]
onboarding_clients = missing_pattern_df[~missing_pattern_df["has_true_gap"]]

print(f"Total unique clients: {missing_pattern_df['client_nr'].nunique()}")
print(f"Clients with genuine data gaps (after first appearance): {len(true_gap_clients)}")
print(f"Clients with only onboarding-style gaps: {len(onboarding_clients)}")
print(f"Total genuine missing month-client records: {sum(len(gaps) for gaps in genuine_gaps_per_client.values())}")
print(f"\nExample clients with genuine gaps:")
for _, row in true_gap_clients.head(5).iterrows():
    gaps = row['missing_months_after_first_observation']
    print(f"  Client {row['client_nr']}: {len(gaps)} missing months - {gaps[:3]}{'...' if len(gaps) > 3 else ''}")

Total unique clients: 992
Clients with genuine data gaps (after first appearance): 38
Clients with only onboarding-style gaps: 954
Total genuine missing month-client records: 210

Example clients with genuine gaps:
  Client 68: 6 missing months - [Period('2014-03', 'M'), Period('2014-06', 'M'), Period('2014-07', 'M')]...
  Client 140: 1 missing months - [Period('2016-04', 'M')]
  Client 173: 1 missing months - [Period('2016-04', 'M')]
  Client 196: 10 missing months - [Period('2014-03', 'M'), Period('2014-05', 'M'), Period('2014-08', 'M')]...
  Client 219: 2 missing months - [Period('2016-06', 'M'), Period('2016-07', 'M')]


## Impute genuine data gaps
For each genuine gap month, build a synthetic row using medians and last-known values computed only from that client's observations before the gap (no data leakage).

In [50]:
# Impute genuine data gaps with values based only on previous observations
# Create imputed rows 

imputed_rows = []

for client_id, gaps in genuine_gaps_per_client.items():
    # Get all data for this client, sorted chronologically
    client_data = merged_data[merged_data['client_nr'] == client_id].sort_values('yearmonth')
    
    for gap_month in gaps:
        # Get all observations BEFORE this gap month (no data leakage)
        previous_data = client_data[client_data['yearmonth'] < gap_month]
        
        if len(previous_data) == 0:
            # No previous data - shouldn't happen since we only track gaps after first observation
            continue
        
        # Calculate medians for numeric columns from previous observations only
        nr_debit_trx_median = previous_data['nr_debit_trx'].median()
        volume_debit_trx_median = previous_data['volume_debit_trx'].median()
        nr_credit_trx_median = previous_data['nr_credit_trx'].median()
        volume_credit_trx_median = previous_data['volume_credit_trx'].median()
        min_balance_median = previous_data['min_balance'].median()
        max_balance_median = previous_data['max_balance'].median()
        
        # Get last known CRG value (forward fill from previous month)
        last_crg = previous_data['CRG'].iloc[-1]
        
        # Create the imputed row with all required columns (without target column)
        imputed_row = {
            'client_nr': client_id,
            'yearmonth': gap_month,
            'credit_application': 0,  # Assume no credit application in gap
            'nr_credit_applications': 0,  # Assume no credit application in gap
            'nr_debit_trx': nr_debit_trx_median,
            'volume_debit_trx': volume_debit_trx_median,
            'nr_credit_trx': nr_credit_trx_median,
            'volume_credit_trx': volume_credit_trx_median,
            'total_nr_trx': nr_credit_trx_median + nr_debit_trx_median,
            'min_balance': min_balance_median,
            'max_balance': max_balance_median,
            'CRG': last_crg,
        }
        
        imputed_rows.append(imputed_row)

# Create base DataFrame from imputed rows (reusable for both t+1 and t+3)
imputed_df = pd.DataFrame(imputed_rows)

print(f"Created {len(imputed_df)} imputed rows for genuine data gaps")
print(f"Covering {len(genuine_gaps_per_client)} clients with {sum(len(gaps) for gaps in genuine_gaps_per_client.values())} missing month-client combinations")
display(imputed_df.head(10))

Created 210 imputed rows for genuine data gaps
Covering 38 clients with 210 missing month-client combinations


,client_nr,yearmonth,credit_application,nr_credit_applications,nr_debit_trx,volume_debit_trx,nr_credit_trx,volume_credit_trx,total_nr_trx,min_balance,max_balance,CRG
0,68,2014-03,0,0,0.00,0.00,1.00,"2,380,600.00",1.00,0.00,0.00,7.00
1,68,2014-06,0,0,1.00,140.00,0.00,0.00,1.00,"-38,137,586.00","-38,136,501.00",7.00
2,68,2014-07,0,0,1.00,140.00,0.00,0.00,1.00,"-38,137,586.00","-38,136,501.00",7.00
3,68,2014-08,0,0,1.00,140.00,0.00,0.00,1.00,"-38,137,586.00","-38,136,501.00",7.00
4,68,2014-09,0,0,1.00,140.00,0.00,0.00,1.00,"-38,137,586.00","-38,136,501.00",7.00
5,68,2014-10,0,0,1.00,140.00,0.00,0.00,1.00,"-38,137,586.00","-38,136,501.00",7.00
6,140,2016-04,0,0,1.00,140.00,0.00,0.00,1.00,"-117,646,118.00","-117,584,128.00",7.00
7,173,2016-04,0,0,1.00,140.00,0.00,0.00,1.00,"-11,771,379.00","-10,698,704.00",7.00
8,196,2014-03,0,0,0.50,39.50,0.50,450.00,1.00,"11,102.00","11,102.00",NaN
9,196,2014-05,0,0,1.00,79.00,0.00,0.00,1.00,"11,394.00","11,394.00",NaN


## Combine original and imputed rows
Concatenate the original data with the imputed gap rows into `merged_data_imputed`, flagging imputed rows via the `imputed` column so they can be excluded from test set evaluation later.

In [51]:
# Imputed rows need to be removed during test set evaluation
imputed_df['imputed'] = True  
merged_data['imputed'] = False  # Mark original rows as not imputed

# Fill missing columns with NaN for columns not in imputed_df_t1
for col in merged_data.columns:
    if col not in imputed_df.columns:
        imputed_df[col] = np.nan

# Reorder columns to match merged_data
imputed_df = imputed_df[merged_data.columns]

# Concatenate original and imputed data
merged_data_imputed = pd.concat([merged_data, imputed_df], ignore_index=True)

# Re-sort by client and month to maintain temporal order
merged_data_imputed = merged_data_imputed.sort_values(['client_nr', 'yearmonth']).reset_index(drop=True)

print(f"  Shape before imputation: {merged_data.shape}")
print(f"  Shape after imputation: {merged_data_imputed.shape}")
print(f"  Rows added: {len(imputed_df)}")
display(merged_data_imputed.head(20))

  Shape before imputation: (29996, 13)
  Shape after imputation: (30206, 13)
  Rows added: 210


,client_nr,yearmonth,credit_application,nr_credit_applications,total_nr_trx,nr_debit_trx,volume_debit_trx,nr_credit_trx,volume_credit_trx,min_balance,max_balance,CRG,imputed
0,1,2014-01,0,0,97.00,50.00,"6,527,929.00",47.00,"7,454,863.00","-7,914,288.00","25,110,651.00",1.00,False
1,1,2014-02,0,0,88.00,59.00,"3,475,918.00",29.00,"1,895,848.00","-8,448,513.00","25,036,651.00",1.00,False
2,1,2014-03,0,0,96.00,62.00,"31,316,405.00",34.00,"20,083,583.00","-10,347,650.00","18,020,151.00",1.00,False
3,1,2014-04,0,0,83.00,53.00,"18,669,967.00",30.00,"1,091,295.00","-15,385,039.00","13,318,200.00",1.00,False
4,1,2014-05,0,0,94.00,54.00,"2,893,905.00",40.00,"2,034,075.00","-15,682,170.00","2,350,000.00",1.00,False
5,1,2014-06,0,0,74.00,51.00,"2,083,142.00",23.00,"3,241,073.00","-15,927,514.00","2,000,000.00",1.00,False
6,1,2014-07,0,0,76.00,59.00,"2,538,771.00",17.00,"4,564,281.00","-15,823,639.00","2,005,161.00",1.00,False
7,1,2014-08,0,0,62.00,40.00,"2,620,143.00",22.00,"4,280,647.00","-14,468,191.00","1,750,000.00",1.00,False
8,1,2014-09,0,0,90.00,49.00,"2,500,177.00",41.00,"8,339,304.00","-12,025,540.00","1,600,000.00",1.00,False
9,1,2014-10,0,0,112.00,68.00,"5,848,714.00",44.00,"17,013,661.00","-7,211,508.00","7,819,451.00",1.00,False


## Feature: tenure
Add `months_history_available`, the number of months since each client's first observed record.

In [52]:
# create duration feature (nr of months since first appearance)
first_month = merged_data_imputed.groupby("client_nr")["yearmonth"].transform("min")

merged_data_imputed["months_history_available"] = (
    merged_data_imputed["yearmonth"].astype("int64") - first_month.astype("int64")
)

In [53]:
merged_data_imputed

,client_nr,yearmonth,credit_application,nr_credit_applications,total_nr_trx,nr_debit_trx,volume_debit_trx,nr_credit_trx,volume_credit_trx,min_balance,max_balance,CRG,imputed,months_history_available
0,1,2014-01,0,0,97.00,50.00,"6,527,929.00",47.00,"7,454,863.00","-7,914,288.00","25,110,651.00",1.00,False,0
1,1,2014-02,0,0,88.00,59.00,"3,475,918.00",29.00,"1,895,848.00","-8,448,513.00","25,036,651.00",1.00,False,1
2,1,2014-03,0,0,96.00,62.00,"31,316,405.00",34.00,"20,083,583.00","-10,347,650.00","18,020,151.00",1.00,False,2
3,1,2014-04,0,0,83.00,53.00,"18,669,967.00",30.00,"1,091,295.00","-15,385,039.00","13,318,200.00",1.00,False,3
4,1,2014-05,0,0,94.00,54.00,"2,893,905.00",40.00,"2,034,075.00","-15,682,170.00","2,350,000.00",1.00,False,4
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
30201,1000,2016-04,0,0,2.00,1.00,"605,000.00",1.00,"315,800.00","131,422.00","736,422.00",NaN,False,26
30202,1000,2016-05,0,0,5.00,3.00,"607,506.00",2.00,"1,210,000.00","128,916.00","735,145.00",NaN,False,27
30203,1000,2016-06,0,0,4.00,3.00,"1,211,270.00",1.00,"605,000.00","127,646.00","1,338,916.00",NaN,False,28
30204,1000,2016-07,0,0,4.00,2.00,"606,253.00",2.00,"920,000.00","441,393.00","1,047,646.00",NaN,False,29


## Feature: z-score and month-over-month delta
For each numeric transaction/balance column, add a `_zscore` column (how many standard deviations the current value is from the client's mean over prior months) and a `_delta` column (percentual change vs. the previous month).

`_zscore` values of `inf`/`-inf` (which occur when a client's prior values had zero variance) are capped at `+5`/`-5` instead of being dropped, so the extreme deviation is still captured as a large but finite value (since 5 is a value most real z-scores wouldn't naturally reach). Remaining `NaN`s (insufficient history) are left as-is here, since `xgboost_data` can handle them natively; they are filled with `0` later, only for `log_reg_data`.

In [54]:
# check how many zero values each numeric column has, since zeros affect delta/ratio-based features
numeric_columns = [
    "total_nr_trx",
    "nr_debit_trx",
    "volume_debit_trx",
    "nr_credit_trx",
    "volume_credit_trx",
    "min_balance",
    "max_balance",
]

zero_counts = (merged_data_imputed[numeric_columns] == 0).sum()
zero_counts

total_nr_trx           0
nr_debit_trx         178
volume_debit_trx     178
nr_credit_trx        892
volume_credit_trx    892
min_balance          339
max_balance          189
dtype: int64

In [55]:
# create feature: nr of stds the current value is from the client's mean over prior months (excludes current month to avoid leakage)
merged_data_imputed = merged_data_imputed.sort_values(['client_nr', 'yearmonth']).reset_index(drop=True)

numeric_columns = [
    "total_nr_trx",
    "nr_debit_trx",
    "volume_debit_trx",
    "nr_credit_trx",
    "volume_credit_trx",
    "min_balance",
    "max_balance",
]

def zscore_vs_past(s):
    past = s.shift(1)
    return (s - past.expanding().mean()) / past.expanding().std()

for col in numeric_columns:
    zscore = merged_data_imputed.groupby("client_nr")[col].transform(zscore_vs_past)
    # cap extreme z-scores at +/-5 (occurs when past values had zero variance); still leaves NaN for insufficient history
    merged_data_imputed[f"{col}_zscore"] = zscore.replace([np.inf, -np.inf], [5, -5])

# create feature: percentual difference of feature with the previous month's value
for col in numeric_columns:
    # replace zeros with a small value to avoid division by zero in pct_change
    non_zero_col = merged_data_imputed[col].replace(0, 0.01)
    merged_data_imputed[f"{col}_delta"] = (
        non_zero_col.groupby(merged_data_imputed["client_nr"]).transform(lambda s: s.pct_change())
    )


In [56]:
merged_data_imputed

,client_nr,yearmonth,credit_application,nr_credit_applications,total_nr_trx,nr_debit_trx,volume_debit_trx,nr_credit_trx,volume_credit_trx,min_balance,max_balance,CRG,imputed,months_history_available,total_nr_trx_zscore,nr_debit_trx_zscore,volume_debit_trx_zscore,nr_credit_trx_zscore,volume_credit_trx_zscore,min_balance_zscore,max_balance_zscore,total_nr_trx_delta,nr_debit_trx_delta,volume_debit_trx_delta,nr_credit_trx_delta,volume_credit_trx_delta,min_balance_delta,max_balance_delta
0,1,2014-01,0,0,97.00,50.00,"6,527,929.00",47.00,"7,454,863.00","-7,914,288.00","25,110,651.00",1.00,False,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1,2014-02,0,0,88.00,59.00,"3,475,918.00",29.00,"1,895,848.00","-8,448,513.00","25,036,651.00",1.00,False,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-0.09,0.18,-0.47,-0.38,-0.75,0.07,-0.00
2,1,2014-03,0,0,96.00,62.00,"31,316,405.00",34.00,"20,083,583.00","-10,347,650.00","18,020,151.00",1.00,False,2,0.55,1.18,12.19,-0.31,3.92,-5.73,-134.80,0.09,0.05,8.01,0.17,9.59,0.22,-0.28
3,1,2014-04,0,0,83.00,53.00,"18,669,967.00",30.00,"1,091,295.00","-15,385,039.00","13,318,200.00",1.00,False,3,-2.16,-0.64,0.32,-0.72,-0.94,-5.07,-2.31,-0.14,-0.15,-0.40,-0.12,-0.95,0.49,-0.26
4,1,2014-05,0,0,94.00,54.00,"2,893,905.00",40.00,"2,034,075.00","-15,682,170.00","2,350,000.00",1.00,False,4,0.45,-0.37,-0.95,0.60,-0.64,-1.51,-3.13,0.13,0.02,-0.84,0.33,0.86,0.02,-0.82
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
30201,1000,2016-04,0,0,2.00,1.00,"605,000.00",1.00,"315,800.00","131,422.00","736,422.00",NaN,False,26,-1.16,-1.29,-0.28,-0.78,-0.90,-0.32,-0.44,-0.60,-0.67,-0.50,-0.50,-0.74,-0.69,-0.55
30202,1000,2016-05,0,0,5.00,3.00,"607,506.00",2.00,"1,210,000.00","128,916.00","735,145.00",NaN,False,27,0.04,0.15,-0.27,-0.08,0.67,-0.32,-0.43,1.50,2.00,0.00,1.00,2.83,-0.02,-0.00
30203,1000,2016-06,0,0,4.00,3.00,"1,211,270.00",1.00,"605,000.00","127,646.00","1,338,916.00",NaN,False,28,-0.35,0.15,0.53,-0.76,-0.40,-0.32,0.66,-0.20,0.00,0.99,-0.50,-0.50,-0.01,0.82
30204,1000,2016-07,0,0,4.00,2.00,"606,253.00",2.00,"920,000.00","441,393.00","1,047,646.00",NaN,False,29,-0.34,-0.57,-0.29,-0.05,0.17,0.75,0.12,0.00,-0.33,-0.50,1.00,0.52,2.46,-0.22


## Feature: past credit application count
Add `nr_credit_applications_past_sum`, the cumulative number of credit applications made in prior months (current month excluded, to avoid leakage).

In [57]:
# create feature: cumulative sum of nr_credit_applications in prior months (excludes current month)
merged_data_imputed["nr_credit_applications_past_sum"] = (
    merged_data_imputed.groupby("client_nr")["nr_credit_applications"].transform(
        lambda s: s.shift(fill_value=0).cumsum()
    )
)

In [58]:
merged_data_imputed

,client_nr,yearmonth,credit_application,nr_credit_applications,total_nr_trx,nr_debit_trx,volume_debit_trx,nr_credit_trx,volume_credit_trx,min_balance,max_balance,CRG,imputed,months_history_available,total_nr_trx_zscore,nr_debit_trx_zscore,volume_debit_trx_zscore,nr_credit_trx_zscore,volume_credit_trx_zscore,min_balance_zscore,max_balance_zscore,total_nr_trx_delta,nr_debit_trx_delta,volume_debit_trx_delta,nr_credit_trx_delta,volume_credit_trx_delta,min_balance_delta,max_balance_delta,nr_credit_applications_past_sum
0,1,2014-01,0,0,97.00,50.00,"6,527,929.00",47.00,"7,454,863.00","-7,914,288.00","25,110,651.00",1.00,False,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0
1,1,2014-02,0,0,88.00,59.00,"3,475,918.00",29.00,"1,895,848.00","-8,448,513.00","25,036,651.00",1.00,False,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-0.09,0.18,-0.47,-0.38,-0.75,0.07,-0.00,0
2,1,2014-03,0,0,96.00,62.00,"31,316,405.00",34.00,"20,083,583.00","-10,347,650.00","18,020,151.00",1.00,False,2,0.55,1.18,12.19,-0.31,3.92,-5.73,-134.80,0.09,0.05,8.01,0.17,9.59,0.22,-0.28,0
3,1,2014-04,0,0,83.00,53.00,"18,669,967.00",30.00,"1,091,295.00","-15,385,039.00","13,318,200.00",1.00,False,3,-2.16,-0.64,0.32,-0.72,-0.94,-5.07,-2.31,-0.14,-0.15,-0.40,-0.12,-0.95,0.49,-0.26,0
4,1,2014-05,0,0,94.00,54.00,"2,893,905.00",40.00,"2,034,075.00","-15,682,170.00","2,350,000.00",1.00,False,4,0.45,-0.37,-0.95,0.60,-0.64,-1.51,-3.13,0.13,0.02,-0.84,0.33,0.86,0.02,-0.82,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
30201,1000,2016-04,0,0,2.00,1.00,"605,000.00",1.00,"315,800.00","131,422.00","736,422.00",NaN,False,26,-1.16,-1.29,-0.28,-0.78,-0.90,-0.32,-0.44,-0.60,-0.67,-0.50,-0.50,-0.74,-0.69,-0.55,0
30202,1000,2016-05,0,0,5.00,3.00,"607,506.00",2.00,"1,210,000.00","128,916.00","735,145.00",NaN,False,27,0.04,0.15,-0.27,-0.08,0.67,-0.32,-0.43,1.50,2.00,0.00,1.00,2.83,-0.02,-0.00,0
30203,1000,2016-06,0,0,4.00,3.00,"1,211,270.00",1.00,"605,000.00","127,646.00","1,338,916.00",NaN,False,28,-0.35,0.15,0.53,-0.76,-0.40,-0.32,0.66,-0.20,0.00,0.99,-0.50,-0.50,-0.01,0.82,0
30204,1000,2016-07,0,0,4.00,2.00,"606,253.00",2.00,"920,000.00","441,393.00","1,047,646.00",NaN,False,29,-0.34,-0.57,-0.29,-0.05,0.17,0.75,0.12,0.00,-0.33,-0.50,1.00,0.52,2.46,-0.22,0


## Feature: months since last credit application
Add `months_since_last_credit_application` (current month excluded, to avoid leakage); `-1` means the client never applied before.

In [59]:
# create feature: nr of months since credit_application was last 1 (excludes current month to avoid leakage)
# -1 indicates no prior credit application for that client
def months_since_last_credit_application(group):
    yearmonth_int = group["yearmonth"].astype("int64")
    last_app_month = yearmonth_int.where(group["credit_application"] == 1).ffill().shift(1)
    return (yearmonth_int - last_app_month).fillna(-1)

merged_data_imputed["months_since_last_credit_application"] = (
    merged_data_imputed.groupby("client_nr", group_keys=False).apply(months_since_last_credit_application)
)

In [60]:
merged_data_imputed

,client_nr,yearmonth,credit_application,nr_credit_applications,total_nr_trx,nr_debit_trx,volume_debit_trx,nr_credit_trx,volume_credit_trx,min_balance,max_balance,CRG,imputed,months_history_available,total_nr_trx_zscore,nr_debit_trx_zscore,volume_debit_trx_zscore,nr_credit_trx_zscore,volume_credit_trx_zscore,min_balance_zscore,max_balance_zscore,total_nr_trx_delta,nr_debit_trx_delta,volume_debit_trx_delta,nr_credit_trx_delta,volume_credit_trx_delta,min_balance_delta,max_balance_delta,nr_credit_applications_past_sum,months_since_last_credit_application
0,1,2014-01,0,0,97.00,50.00,"6,527,929.00",47.00,"7,454,863.00","-7,914,288.00","25,110,651.00",1.00,False,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,-1.00
1,1,2014-02,0,0,88.00,59.00,"3,475,918.00",29.00,"1,895,848.00","-8,448,513.00","25,036,651.00",1.00,False,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-0.09,0.18,-0.47,-0.38,-0.75,0.07,-0.00,0,-1.00
2,1,2014-03,0,0,96.00,62.00,"31,316,405.00",34.00,"20,083,583.00","-10,347,650.00","18,020,151.00",1.00,False,2,0.55,1.18,12.19,-0.31,3.92,-5.73,-134.80,0.09,0.05,8.01,0.17,9.59,0.22,-0.28,0,-1.00
3,1,2014-04,0,0,83.00,53.00,"18,669,967.00",30.00,"1,091,295.00","-15,385,039.00","13,318,200.00",1.00,False,3,-2.16,-0.64,0.32,-0.72,-0.94,-5.07,-2.31,-0.14,-0.15,-0.40,-0.12,-0.95,0.49,-0.26,0,-1.00
4,1,2014-05,0,0,94.00,54.00,"2,893,905.00",40.00,"2,034,075.00","-15,682,170.00","2,350,000.00",1.00,False,4,0.45,-0.37,-0.95,0.60,-0.64,-1.51,-3.13,0.13,0.02,-0.84,0.33,0.86,0.02,-0.82,0,-1.00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
30201,1000,2016-04,0,0,2.00,1.00,"605,000.00",1.00,"315,800.00","131,422.00","736,422.00",NaN,False,26,-1.16,-1.29,-0.28,-0.78,-0.90,-0.32,-0.44,-0.60,-0.67,-0.50,-0.50,-0.74,-0.69,-0.55,0,-1.00
30202,1000,2016-05,0,0,5.00,3.00,"607,506.00",2.00,"1,210,000.00","128,916.00","735,145.00",NaN,False,27,0.04,0.15,-0.27,-0.08,0.67,-0.32,-0.43,1.50,2.00,0.00,1.00,2.83,-0.02,-0.00,0,-1.00
30203,1000,2016-06,0,0,4.00,3.00,"1,211,270.00",1.00,"605,000.00","127,646.00","1,338,916.00",NaN,False,28,-0.35,0.15,0.53,-0.76,-0.40,-0.32,0.66,-0.20,0.00,0.99,-0.50,-0.50,-0.01,0.82,0,-1.00
30204,1000,2016-07,0,0,4.00,2.00,"606,253.00",2.00,"920,000.00","441,393.00","1,047,646.00",NaN,False,29,-0.34,-0.57,-0.29,-0.05,0.17,0.75,0.12,0.00,-0.33,-0.50,1.00,0.52,2.46,-0.22,0,-1.00


## Feature: calendar month
Add `month`, the calendar month (1-12) of the observation, so the models can pick up seasonality in credit application behaviour.

In [61]:
merged_data_imputed["month"] = merged_data_imputed["yearmonth"].dt.month

## Feature: 3-month balance trend
Add `{col}_slope_3m` for `min_balance` and `max_balance`: the slope of a linear fit over the last three months, normalized by the client's mean absolute balance in that window so the trend is comparable across clients of different sizes.

In [62]:
# create feature: normalized 3-month trend in balances (declining balance can signal credit need)
SLOPE_WINDOW = 3
balance_columns = ["min_balance", "max_balance"]

def normalized_rolling_slope(series, window=SLOPE_WINDOW):
    x = np.arange(window)

    def slope(y):
        # scale by the window's average magnitude so slopes are comparable across clients
        scale = np.abs(y).mean()
        return np.polyfit(x, y, 1)[0] / scale if scale else 0.0

    return series.rolling(window).apply(slope, raw=True)

for col in balance_columns:
    merged_data_imputed[f"{col}_slope_3m"] = (
        merged_data_imputed.groupby("client_nr")[col].transform(normalized_rolling_slope)
    )

## Build model-specific input datasets
Create `xgboost_data` (CRG and `month` kept as-is, since tree models handle categoricals natively) and `log_reg_data` (CRG and `month` one-hot encoded, with a `CRG_missing` column for missing values). `month` needs one-hot encoding for logistic regression because its integer values are cyclical, not ordinal. Since logistic regression cannot handle `NaN` values, the remaining `NaN`s in the `_zscore`/`_delta` features are also filled with `0` for `log_reg_data` only (safe to do, since `months_history_available` already flags insufficient history).

In [63]:
# copy for xgboost input (tree models can handle CRG and month as-is)
xgboost_data = merged_data_imputed.copy()

# copy for logistic regression input: one-hot encode CRG, with a dedicated column for missing values
log_reg_data = merged_data_imputed.copy()
crg_dummies = pd.get_dummies(log_reg_data["CRG"], prefix="CRG", dummy_na=True)
crg_dummies = crg_dummies.rename(columns={"CRG_nan": "CRG_missing"}).astype(int)
log_reg_data = pd.concat(
    [log_reg_data.drop(columns=["CRG"]), crg_dummies], axis=1
)

# one-hot encode month as well, since its integer values are cyclical rather than ordinal
month_dummies = pd.get_dummies(log_reg_data["month"], prefix="month").astype(int)
log_reg_data = pd.concat(
    [log_reg_data.drop(columns=["month"]), month_dummies], axis=1
)

# logistic regression cannot handle NaN, unlike tree-based models, so fill remaining NaNs in the zscore/delta features
# NaNs are safe to fill with 0 since months_history_available already flags insufficient history
for col in numeric_columns:
    log_reg_data[f"{col}_zscore"] = log_reg_data[f"{col}_zscore"].fillna(0)
    log_reg_data[f"{col}_delta"] = log_reg_data[f"{col}_delta"].fillna(0)
for col in balance_columns:
    log_reg_data[f"{col}_slope_3m"] = log_reg_data[f"{col}_slope_3m"].fillna(0)

In [64]:
log_reg_data

,client_nr,yearmonth,credit_application,nr_credit_applications,total_nr_trx,nr_debit_trx,volume_debit_trx,nr_credit_trx,volume_credit_trx,min_balance,max_balance,imputed,months_history_available,total_nr_trx_zscore,nr_debit_trx_zscore,volume_debit_trx_zscore,nr_credit_trx_zscore,volume_credit_trx_zscore,min_balance_zscore,max_balance_zscore,total_nr_trx_delta,nr_debit_trx_delta,volume_debit_trx_delta,nr_credit_trx_delta,volume_credit_trx_delta,min_balance_delta,max_balance_delta,nr_credit_applications_past_sum,months_since_last_credit_application,min_balance_slope_3m,max_balance_slope_3m,CRG_1.0,CRG_2.0,CRG_3.0,CRG_4.0,CRG_5.0,CRG_7.0,CRG_missing,month_1,month_2,month_3,month_4,month_5,month_6,month_7,month_8,month_9,month_10,month_11,month_12
0,1,2014-01,0,0,97.00,50.00,"6,527,929.00",47.00,"7,454,863.00","-7,914,288.00","25,110,651.00",False,0,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0,-1.00,0.00,0.00,1,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0
1,1,2014-02,0,0,88.00,59.00,"3,475,918.00",29.00,"1,895,848.00","-8,448,513.00","25,036,651.00",False,1,0.00,0.00,0.00,0.00,0.00,0.00,0.00,-0.09,0.18,-0.47,-0.38,-0.75,0.07,-0.00,0,-1.00,0.00,0.00,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0
2,1,2014-03,0,0,96.00,62.00,"31,316,405.00",34.00,"20,083,583.00","-10,347,650.00","18,020,151.00",False,2,0.55,1.18,12.19,-0.31,3.92,-5.73,-134.80,0.09,0.05,8.01,0.17,9.59,0.22,-0.28,0,-1.00,-0.14,-0.16,1,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0
3,1,2014-04,0,0,83.00,53.00,"18,669,967.00",30.00,"1,091,295.00","-15,385,039.00","13,318,200.00",False,3,-2.16,-0.64,0.32,-0.72,-0.94,-5.07,-2.31,-0.14,-0.15,-0.40,-0.12,-0.95,0.49,-0.26,0,-1.00,-0.30,-0.31,1,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0
4,1,2014-05,0,0,94.00,54.00,"2,893,905.00",40.00,"2,034,075.00","-15,682,170.00","2,350,000.00",False,4,0.45,-0.37,-0.95,0.60,-0.64,-1.51,-3.13,0.13,0.02,-0.84,0.33,0.86,0.02,-0.82,0,-1.00,-0.19,-0.70,1,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
30201,1000,2016-04,0,0,2.00,1.00,"605,000.00",1.00,"315,800.00","131,422.00","736,422.00",False,26,-1.16,-1.29,-0.28,-0.78,-0.90,-0.32,-0.44,-0.60,-0.67,-0.50,-0.50,-0.74,-0.69,-0.55,0,-1.00,-0.45,-0.34,0,0,0,0,0,0,1,0,0,0,1,0,0,0,0,0,0,0,0
30202,1000,2016-05,0,0,5.00,3.00,"607,506.00",2.00,"1,210,000.00","128,916.00","735,145.00",False,27,0.04,0.15,-0.27,-0.08,0.67,-0.32,-0.43,1.50,2.00,0.00,1.00,2.83,-0.02,-0.00,0,-1.00,-0.64,-0.43,0,0,0,0,0,0,1,0,0,0,0,1,0,0,0,0,0,0,0
30203,1000,2016-06,0,0,4.00,3.00,"1,211,270.00",1.00,"605,000.00","127,646.00","1,338,916.00",False,28,-0.35,0.15,0.53,-0.76,-0.40,-0.32,0.66,-0.20,0.00,0.99,-0.50,-0.50,-0.01,0.82,0,-1.00,-0.01,0.32,0,0,0,0,0,0,1,0,0,0,0,0,1,0,0,0,0,0,0
30204,1000,2016-07,0,0,4.00,2.00,"606,253.00",2.00,"920,000.00","441,393.00","1,047,646.00",False,29,-0.34,-0.57,-0.29,-0.05,0.17,0.75,0.12,0.00,-0.33,-0.50,1.00,0.52,2.46,-0.22,0,-1.00,0.67,0.15,0,0,0,0,0,0,1,0,0,0,0,0,0,1,0,0,0,0,0


In [65]:
# check for columns with remaining NaN or inf/-inf values in log_reg_data
numeric_log_reg_columns = log_reg_data.select_dtypes(include=[np.number]).columns

nan_counts = log_reg_data[numeric_log_reg_columns].isna().sum()
inf_counts = np.isinf(log_reg_data[numeric_log_reg_columns]).sum()

assert nan_counts.sum() == 0, f"Found NaN values in columns: {nan_counts[nan_counts > 0].to_dict()}"
assert inf_counts.sum() == 0, f"Found inf/-inf values in columns: {inf_counts[inf_counts > 0].to_dict()}"

pd.DataFrame({"nan_count": nan_counts, "inf_count": inf_counts}).loc[
    lambda df: (df["nan_count"] > 0) | (df["inf_count"] > 0)
]

,nan_count,inf_count


## Create prediction targets
For both `xgboost_data` and `log_reg_data`, create a t+1 and a t+3 target (`credit_application` shifted 1 and 3 months ahead per client), dropping rows without a future value. This yields four final datasets: `xgboost_data_t1`, `log_reg_data_t1`, `xgboost_data_t3`, `log_reg_data_t3`.

In [66]:
# Create target variables: credit_application value for the same client N months ahead, for N=[1,3]
datasets_t1 = {}
datasets_t3 = {}

for name, base_df in [("xgboost_data", xgboost_data), ("log_reg_data", log_reg_data)]:
    df_t1 = base_df.copy()
    df_t1['target_t+1'] = df_t1.groupby('client_nr')['credit_application'].shift(-1)
    df_t1.dropna(subset=['target_t+1'], inplace=True)
    datasets_t1[name] = df_t1

    df_t3 = base_df.copy()
    df_t3['target_t+3'] = df_t3.groupby('client_nr')['credit_application'].shift(-3)
    df_t3.dropna(subset=['target_t+3'], inplace=True)
    datasets_t3[name] = df_t3

xgboost_data_t1 = datasets_t1["xgboost_data"]
log_reg_data_t1 = datasets_t1["log_reg_data"]
xgboost_data_t3 = datasets_t3["xgboost_data"]
log_reg_data_t3 = datasets_t3["log_reg_data"]

display(xgboost_data_t1.head(10))
display(log_reg_data_t1.head(10))
display(xgboost_data_t3.head(10))
display(log_reg_data_t3.head(10))

,client_nr,yearmonth,credit_application,nr_credit_applications,total_nr_trx,nr_debit_trx,volume_debit_trx,nr_credit_trx,volume_credit_trx,min_balance,max_balance,CRG,imputed,months_history_available,total_nr_trx_zscore,nr_debit_trx_zscore,volume_debit_trx_zscore,nr_credit_trx_zscore,volume_credit_trx_zscore,min_balance_zscore,max_balance_zscore,total_nr_trx_delta,nr_debit_trx_delta,volume_debit_trx_delta,nr_credit_trx_delta,volume_credit_trx_delta,min_balance_delta,max_balance_delta,nr_credit_applications_past_sum,months_since_last_credit_application,month,min_balance_slope_3m,max_balance_slope_3m,target_t+1
0,1,2014-01,0,0,97.00,50.00,"6,527,929.00",47.00,"7,454,863.00","-7,914,288.00","25,110,651.00",1.00,False,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,-1.00,1,NaN,NaN,0.00
1,1,2014-02,0,0,88.00,59.00,"3,475,918.00",29.00,"1,895,848.00","-8,448,513.00","25,036,651.00",1.00,False,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-0.09,0.18,-0.47,-0.38,-0.75,0.07,-0.00,0,-1.00,2,NaN,NaN,0.00
2,1,2014-03,0,0,96.00,62.00,"31,316,405.00",34.00,"20,083,583.00","-10,347,650.00","18,020,151.00",1.00,False,2,0.55,1.18,12.19,-0.31,3.92,-5.73,-134.80,0.09,0.05,8.01,0.17,9.59,0.22,-0.28,0,-1.00,3,-0.14,-0.16,0.00
3,1,2014-04,0,0,83.00,53.00,"18,669,967.00",30.00,"1,091,295.00","-15,385,039.00","13,318,200.00",1.00,False,3,-2.16,-0.64,0.32,-0.72,-0.94,-5.07,-2.31,-0.14,-0.15,-0.40,-0.12,-0.95,0.49,-0.26,0,-1.00,4,-0.30,-0.31,0.00
4,1,2014-05,0,0,94.00,54.00,"2,893,905.00",40.00,"2,034,075.00","-15,682,170.00","2,350,000.00",1.00,False,4,0.45,-0.37,-0.95,0.60,-0.64,-1.51,-3.13,0.13,0.02,-0.84,0.33,0.86,0.02,-0.82,0,-1.00,5,-0.19,-0.70,0.00
5,1,2014-06,0,0,74.00,51.00,"2,083,142.00",23.00,"3,241,073.00","-15,927,514.00","2,000,000.00",1.00,False,5,-2.96,-0.95,-0.86,-1.73,-0.41,-1.17,-1.56,-0.21,-0.06,-0.28,-0.43,0.59,0.02,-0.15,0,-1.00,6,-0.02,-0.96,0.00
6,1,2014-07,0,0,76.00,59.00,"2,538,771.00",17.00,"4,564,281.00","-15,823,639.00","2,005,161.00",1.00,False,6,-1.42,0.88,-0.70,-1.97,-0.19,-0.93,-1.18,0.03,0.16,0.22,-0.26,0.41,-0.01,0.00,0,-1.00,7,-0.00,-0.08,0.00
7,1,2014-08,0,0,62.00,40.00,"2,620,143.00",22.00,"4,280,647.00","-14,468,191.00","1,750,000.00",1.00,False,7,-2.63,-3.37,-0.63,-0.94,-0.22,-0.45,-1.02,-0.18,-0.32,0.03,0.29,-0.06,-0.09,-0.13,0,-1.00,8,0.05,-0.07,0.00
8,1,2014-09,0,0,90.00,49.00,"2,500,177.00",41.00,"8,339,304.00","-12,025,540.00","1,600,000.00",1.00,False,8,0.50,-0.65,-0.59,1.08,0.45,0.28,-0.91,0.45,0.23,-0.05,0.86,0.95,-0.17,-0.09,0,-1.00,9,0.13,-0.11,0.00
9,1,2014-10,0,0,112.00,68.00,"5,848,714.00",44.00,"17,013,661.00","-7,211,508.00","7,819,451.00",1.00,False,9,2.34,2.26,-0.22,1.26,1.90,1.73,-0.22,0.24,0.39,1.34,0.07,1.04,-0.40,3.89,0,-1.00,10,0.32,0.82,0.00


,client_nr,yearmonth,credit_application,nr_credit_applications,total_nr_trx,nr_debit_trx,volume_debit_trx,nr_credit_trx,volume_credit_trx,min_balance,max_balance,imputed,months_history_available,total_nr_trx_zscore,nr_debit_trx_zscore,volume_debit_trx_zscore,nr_credit_trx_zscore,volume_credit_trx_zscore,min_balance_zscore,max_balance_zscore,total_nr_trx_delta,nr_debit_trx_delta,volume_debit_trx_delta,nr_credit_trx_delta,volume_credit_trx_delta,min_balance_delta,max_balance_delta,nr_credit_applications_past_sum,months_since_last_credit_application,min_balance_slope_3m,max_balance_slope_3m,CRG_1.0,CRG_2.0,CRG_3.0,CRG_4.0,CRG_5.0,CRG_7.0,CRG_missing,month_1,month_2,month_3,month_4,month_5,month_6,month_7,month_8,month_9,month_10,month_11,month_12,target_t+1
0,1,2014-01,0,0,97.00,50.00,"6,527,929.00",47.00,"7,454,863.00","-7,914,288.00","25,110,651.00",False,0,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0,-1.00,0.00,0.00,1,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0.00
1,1,2014-02,0,0,88.00,59.00,"3,475,918.00",29.00,"1,895,848.00","-8,448,513.00","25,036,651.00",False,1,0.00,0.00,0.00,0.00,0.00,0.00,0.00,-0.09,0.18,-0.47,-0.38,-0.75,0.07,-0.00,0,-1.00,0.00,0.00,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0.00
2,1,2014-03,0,0,96.00,62.00,"31,316,405.00",34.00,"20,083,583.00","-10,347,650.00","18,020,151.00",False,2,0.55,1.18,12.19,-0.31,3.92,-5.73,-134.80,0.09,0.05,8.01,0.17,9.59,0.22,-0.28,0,-1.00,-0.14,-0.16,1,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0.00
3,1,2014-04,0,0,83.00,53.00,"18,669,967.00",30.00,"1,091,295.00","-15,385,039.00","13,318,200.00",False,3,-2.16,-0.64,0.32,-0.72,-0.94,-5.07,-2.31,-0.14,-0.15,-0.40,-0.12,-0.95,0.49,-0.26,0,-1.00,-0.30,-0.31,1,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0.00
4,1,2014-05,0,0,94.00,54.00,"2,893,905.00",40.00,"2,034,075.00","-15,682,170.00","2,350,000.00",False,4,0.45,-0.37,-0.95,0.60,-0.64,-1.51,-3.13,0.13,0.02,-0.84,0.33,0.86,0.02,-0.82,0,-1.00,-0.19,-0.70,1,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0.00
5,1,2014-06,0,0,74.00,51.00,"2,083,142.00",23.00,"3,241,073.00","-15,927,514.00","2,000,000.00",False,5,-2.96,-0.95,-0.86,-1.73,-0.41,-1.17,-1.56,-0.21,-0.06,-0.28,-0.43,0.59,0.02,-0.15,0,-1.00,-0.02,-0.96,1,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0.00
6,1,2014-07,0,0,76.00,59.00,"2,538,771.00",17.00,"4,564,281.00","-15,823,639.00","2,005,161.00",False,6,-1.42,0.88,-0.70,-1.97,-0.19,-0.93,-1.18,0.03,0.16,0.22,-0.26,0.41,-0.01,0.00,0,-1.00,-0.00,-0.08,1,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0.00
7,1,2014-08,0,0,62.00,40.00,"2,620,143.00",22.00,"4,280,647.00","-14,468,191.00","1,750,000.00",False,7,-2.63,-3.37,-0.63,-0.94,-0.22,-0.45,-1.02,-0.18,-0.32,0.03,0.29,-0.06,-0.09,-0.13,0,-1.00,0.05,-0.07,1,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0.00
8,1,2014-09,0,0,90.00,49.00,"2,500,177.00",41.00,"8,339,304.00","-12,025,540.00","1,600,000.00",False,8,0.50,-0.65,-0.59,1.08,0.45,0.28,-0.91,0.45,0.23,-0.05,0.86,0.95,-0.17,-0.09,0,-1.00,0.13,-0.11,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0.00
9,1,2014-10,0,0,112.00,68.00,"5,848,714.00",44.00,"17,013,661.00","-7,211,508.00","7,819,451.00",False,9,2.34,2.26,-0.22,1.26,1.90,1.73,-0.22,0.24,0.39,1.34,0.07,1.04,-0.40,3.89,0,-1.00,0.32,0.82,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0.00


,client_nr,yearmonth,credit_application,nr_credit_applications,total_nr_trx,nr_debit_trx,volume_debit_trx,nr_credit_trx,volume_credit_trx,min_balance,max_balance,CRG,imputed,months_history_available,total_nr_trx_zscore,nr_debit_trx_zscore,volume_debit_trx_zscore,nr_credit_trx_zscore,volume_credit_trx_zscore,min_balance_zscore,max_balance_zscore,total_nr_trx_delta,nr_debit_trx_delta,volume_debit_trx_delta,nr_credit_trx_delta,volume_credit_trx_delta,min_balance_delta,max_balance_delta,nr_credit_applications_past_sum,months_since_last_credit_application,month,min_balance_slope_3m,max_balance_slope_3m,target_t+3
0,1,2014-01,0,0,97.00,50.00,"6,527,929.00",47.00,"7,454,863.00","-7,914,288.00","25,110,651.00",1.00,False,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,-1.00,1,NaN,NaN,0.00
1,1,2014-02,0,0,88.00,59.00,"3,475,918.00",29.00,"1,895,848.00","-8,448,513.00","25,036,651.00",1.00,False,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-0.09,0.18,-0.47,-0.38,-0.75,0.07,-0.00,0,-1.00,2,NaN,NaN,0.00
2,1,2014-03,0,0,96.00,62.00,"31,316,405.00",34.00,"20,083,583.00","-10,347,650.00","18,020,151.00",1.00,False,2,0.55,1.18,12.19,-0.31,3.92,-5.73,-134.80,0.09,0.05,8.01,0.17,9.59,0.22,-0.28,0,-1.00,3,-0.14,-0.16,0.00
3,1,2014-04,0,0,83.00,53.00,"18,669,967.00",30.00,"1,091,295.00","-15,385,039.00","13,318,200.00",1.00,False,3,-2.16,-0.64,0.32,-0.72,-0.94,-5.07,-2.31,-0.14,-0.15,-0.40,-0.12,-0.95,0.49,-0.26,0,-1.00,4,-0.30,-0.31,0.00
4,1,2014-05,0,0,94.00,54.00,"2,893,905.00",40.00,"2,034,075.00","-15,682,170.00","2,350,000.00",1.00,False,4,0.45,-0.37,-0.95,0.60,-0.64,-1.51,-3.13,0.13,0.02,-0.84,0.33,0.86,0.02,-0.82,0,-1.00,5,-0.19,-0.70,0.00
5,1,2014-06,0,0,74.00,51.00,"2,083,142.00",23.00,"3,241,073.00","-15,927,514.00","2,000,000.00",1.00,False,5,-2.96,-0.95,-0.86,-1.73,-0.41,-1.17,-1.56,-0.21,-0.06,-0.28,-0.43,0.59,0.02,-0.15,0,-1.00,6,-0.02,-0.96,0.00
6,1,2014-07,0,0,76.00,59.00,"2,538,771.00",17.00,"4,564,281.00","-15,823,639.00","2,005,161.00",1.00,False,6,-1.42,0.88,-0.70,-1.97,-0.19,-0.93,-1.18,0.03,0.16,0.22,-0.26,0.41,-0.01,0.00,0,-1.00,7,-0.00,-0.08,0.00
7,1,2014-08,0,0,62.00,40.00,"2,620,143.00",22.00,"4,280,647.00","-14,468,191.00","1,750,000.00",1.00,False,7,-2.63,-3.37,-0.63,-0.94,-0.22,-0.45,-1.02,-0.18,-0.32,0.03,0.29,-0.06,-0.09,-0.13,0,-1.00,8,0.05,-0.07,0.00
8,1,2014-09,0,0,90.00,49.00,"2,500,177.00",41.00,"8,339,304.00","-12,025,540.00","1,600,000.00",1.00,False,8,0.50,-0.65,-0.59,1.08,0.45,0.28,-0.91,0.45,0.23,-0.05,0.86,0.95,-0.17,-0.09,0,-1.00,9,0.13,-0.11,0.00
9,1,2014-10,0,0,112.00,68.00,"5,848,714.00",44.00,"17,013,661.00","-7,211,508.00","7,819,451.00",1.00,False,9,2.34,2.26,-0.22,1.26,1.90,1.73,-0.22,0.24,0.39,1.34,0.07,1.04,-0.40,3.89,0,-1.00,10,0.32,0.82,0.00


,client_nr,yearmonth,credit_application,nr_credit_applications,total_nr_trx,nr_debit_trx,volume_debit_trx,nr_credit_trx,volume_credit_trx,min_balance,max_balance,imputed,months_history_available,total_nr_trx_zscore,nr_debit_trx_zscore,volume_debit_trx_zscore,nr_credit_trx_zscore,volume_credit_trx_zscore,min_balance_zscore,max_balance_zscore,total_nr_trx_delta,nr_debit_trx_delta,volume_debit_trx_delta,nr_credit_trx_delta,volume_credit_trx_delta,min_balance_delta,max_balance_delta,nr_credit_applications_past_sum,months_since_last_credit_application,min_balance_slope_3m,max_balance_slope_3m,CRG_1.0,CRG_2.0,CRG_3.0,CRG_4.0,CRG_5.0,CRG_7.0,CRG_missing,month_1,month_2,month_3,month_4,month_5,month_6,month_7,month_8,month_9,month_10,month_11,month_12,target_t+3
0,1,2014-01,0,0,97.00,50.00,"6,527,929.00",47.00,"7,454,863.00","-7,914,288.00","25,110,651.00",False,0,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0,-1.00,0.00,0.00,1,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0.00
1,1,2014-02,0,0,88.00,59.00,"3,475,918.00",29.00,"1,895,848.00","-8,448,513.00","25,036,651.00",False,1,0.00,0.00,0.00,0.00,0.00,0.00,0.00,-0.09,0.18,-0.47,-0.38,-0.75,0.07,-0.00,0,-1.00,0.00,0.00,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0.00
2,1,2014-03,0,0,96.00,62.00,"31,316,405.00",34.00,"20,083,583.00","-10,347,650.00","18,020,151.00",False,2,0.55,1.18,12.19,-0.31,3.92,-5.73,-134.80,0.09,0.05,8.01,0.17,9.59,0.22,-0.28,0,-1.00,-0.14,-0.16,1,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0.00
3,1,2014-04,0,0,83.00,53.00,"18,669,967.00",30.00,"1,091,295.00","-15,385,039.00","13,318,200.00",False,3,-2.16,-0.64,0.32,-0.72,-0.94,-5.07,-2.31,-0.14,-0.15,-0.40,-0.12,-0.95,0.49,-0.26,0,-1.00,-0.30,-0.31,1,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0.00
4,1,2014-05,0,0,94.00,54.00,"2,893,905.00",40.00,"2,034,075.00","-15,682,170.00","2,350,000.00",False,4,0.45,-0.37,-0.95,0.60,-0.64,-1.51,-3.13,0.13,0.02,-0.84,0.33,0.86,0.02,-0.82,0,-1.00,-0.19,-0.70,1,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0.00
5,1,2014-06,0,0,74.00,51.00,"2,083,142.00",23.00,"3,241,073.00","-15,927,514.00","2,000,000.00",False,5,-2.96,-0.95,-0.86,-1.73,-0.41,-1.17,-1.56,-0.21,-0.06,-0.28,-0.43,0.59,0.02,-0.15,0,-1.00,-0.02,-0.96,1,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0.00
6,1,2014-07,0,0,76.00,59.00,"2,538,771.00",17.00,"4,564,281.00","-15,823,639.00","2,005,161.00",False,6,-1.42,0.88,-0.70,-1.97,-0.19,-0.93,-1.18,0.03,0.16,0.22,-0.26,0.41,-0.01,0.00,0,-1.00,-0.00,-0.08,1,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0.00
7,1,2014-08,0,0,62.00,40.00,"2,620,143.00",22.00,"4,280,647.00","-14,468,191.00","1,750,000.00",False,7,-2.63,-3.37,-0.63,-0.94,-0.22,-0.45,-1.02,-0.18,-0.32,0.03,0.29,-0.06,-0.09,-0.13,0,-1.00,0.05,-0.07,1,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0.00
8,1,2014-09,0,0,90.00,49.00,"2,500,177.00",41.00,"8,339,304.00","-12,025,540.00","1,600,000.00",False,8,0.50,-0.65,-0.59,1.08,0.45,0.28,-0.91,0.45,0.23,-0.05,0.86,0.95,-0.17,-0.09,0,-1.00,0.13,-0.11,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0.00
9,1,2014-10,0,0,112.00,68.00,"5,848,714.00",44.00,"17,013,661.00","-7,211,508.00","7,819,451.00",False,9,2.34,2.26,-0.22,1.26,1.90,1.73,-0.22,0.24,0.39,1.34,0.07,1.04,-0.40,3.89,0,-1.00,0.32,0.82,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0.00


In [67]:
# check if the maximum 'yearmonth' is consistent across different datasets
print(xgboost_data_t1['yearmonth'].max())
print(xgboost_data_t3['yearmonth'].max())
print(log_reg_data_t1['yearmonth'].max())
print(log_reg_data_t3['yearmonth'].max())

2016-07
2016-05
2016-07
2016-05


# Trans/Val/Test split

train: yearmonth <= 2015-08 

validation: 2015-09 <= yearmonth <= 2015-12 

test: yearmonth >= 2016-01 (and remove imputed records)

In [68]:
# Split boundaries (inclusive) based on yearmonth
train_end = pd.Period("2015-08", freq="M")
val_start = pd.Period("2015-09", freq="M")
val_end = pd.Period("2015-12", freq="M")
test_start = pd.Period("2016-01", freq="M")

def split_train_val_test(df):
    train = df[df["yearmonth"] <= train_end]
    val = df[(df["yearmonth"] >= val_start) & (df["yearmonth"] <= val_end)]
    # exclude imputed rows from the test set, since they are synthetic and would distort evaluation
    test = df[(df["yearmonth"] >= test_start) & (~df["imputed"])]

    # the imputed flag is no longer needed once the splits are made
    train = train.drop(columns=["imputed"])
    val = val.drop(columns=["imputed"])
    test = test.drop(columns=["imputed"])
    return train, val, test

# datasets to split, keyed by an interpretable name used for the folder structure
datasets_to_split = {
    "xgboost_t1": xgboost_data_t1,
    "xgboost_t3": xgboost_data_t3,
    "log_reg_t1": log_reg_data_t1,
    "log_reg_t3": log_reg_data_t3,
}

splits_by_dataset = {}
for dataset_name, df in datasets_to_split.items():
    train, val, test = split_train_val_test(df)
    splits_by_dataset[dataset_name] = {"train": train, "val": val, "test": test}
    print(f"{dataset_name}: train={len(train)}, val={len(val)}, test={len(test)}")

xgboost_t1: train=18835, val=3790, test=6544
xgboost_t3: train=18803, val=3763, test=4646
log_reg_t1: train=18835, val=3790, test=6544
log_reg_t3: train=18803, val=3763, test=4646


## Save splits
Save each dataset/split combination as Parquet (columnar, compressed, fast to load) under `data/processed/{dataset_name}/{split}.parquet`, so a model training script can load exactly the data it needs by dataset name and split.

In [69]:
processed_data_dir = Path("data/processed")

for dataset_name, splits in splits_by_dataset.items():
    dataset_dir = processed_data_dir / dataset_name
    dataset_dir.mkdir(parents=True, exist_ok=True)

    for split_name, split_df in splits.items():
        # yearmonth is a Period, which parquet cannot store directly; persist it as a plain string instead
        split_df = split_df.assign(yearmonth=split_df["yearmonth"].astype(str))
        split_df.to_parquet(dataset_dir / f"{split_name}.parquet", index=False)

print(f"Saved datasets to {processed_data_dir.resolve()}")

Saved datasets to C:\Users\SVisbee\Repos\Credit_prediction_case\data\processed
